# Qwen Family Testing on Kaggle
This notebook is specifically tailored to test the Qwen Family models.

In [ ]:
"""
================================================================================
KAGGLE CELL 1: Environment Setup & Installations
Run this cell first to install all necessary packages for the Qwen models.
================================================================================
"""
!pip install transformers accelerate pillow bitsandbytes qwen-vl-utils --upgrade


In [ ]:
import torch
import gc
from PIL import Image

# A helper function to prevent Kaggle from crashing due to out-of-memory (OOM) errors.
def clear_vram():
    """Wipes the GPU memory cleanly between model tests."""
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM Cleared.")

# Load your test image (Upload your image to your Kaggle Input/Working directory)
TEST_IMAGE_PATH = "/kaggle/working/log book.png"
TEST_PROMPT = "Extract all handwritten text verbatim and format it."


In [ ]:
"""
================================================================================
KAGGLE CELL 2: The Qwen Series testing function
Alibaba's SOTA models. We load them in 4-bit/8-bit to fit the Kaggle T4.
================================================================================
"""
def test_qwen_family(image_path, model_id="Qwen/Qwen2.5-VL-3B-Instruct"):
    print(f"\n--- Loading {model_id} ---")
    
    # T4 GPUs only support float16, not bfloat16!
    load_kwargs = {"device_map": "auto", "torch_dtype": torch.float16}
    
    # Check if the model is large (7B or 8B parameter). 
    # Note: We must exclude '0.8B' models from matching '8B'
    is_large_model = "7B" in model_id or ("8B" in model_id and "0.8B" not in model_id)
    
    if is_large_model:
        from transformers import BitsAndBytesConfig
        # Use BitsAndBytesConfig instead of deprecated load_in_4bit kwarg to avoid remote code errors
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
        load_kwargs["quantization_config"] = quantization_config

    is_vl = "VL" in model_id

    if is_vl:
        if "2.5" in model_id:
            from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
        else:
            from transformers import Qwen2VLForConditionalGeneration as ModelClass
    else:
        from transformers import AutoModelForCausalLM as ModelClass

    model = ModelClass.from_pretrained(model_id, trust_remote_code=True, **load_kwargs)
    
    if is_vl:
        from transformers import AutoProcessor
        from qwen_vl_utils import process_vision_info
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": TEST_PROMPT},
                ],
            }
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
        ).to("cuda", torch.float16)
    else:
        from transformers import AutoTokenizer
        print(f"Warning: {model_id} is a text-only model! Passing prompt without image.")
        processor = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        messages = [{"role": "user", "content": TEST_PROMPT}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor([text], return_tensors="pt", padding=True).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    print(f"{model_id} Output:\n", output_text[0])

    del model, processor, inputs
    clear_vram()


In [ ]:
test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen2.5-VL-3B-Instruct")

In [ ]:
test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen3.5-0.8B")

In [ ]:
test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen3-8B")